# ESG 텍스트마이닝 — 단계 II: 빈도분석

- 기준: v14 업데이트 반영
- 핵심 변경: Track B strict compact exact 매칭(공백/underscore 제거), bigram+trigram 병행, partial 분리 저장


## Cell 1. 라이브러리 임포트

In [1]:
import ast
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

print("라이브러리 임포트 완료")

라이브러리 임포트 완료


## Cell 2. 경로 설정

In [2]:
# ── 입력 경로 ──────────────────────────────────────────────────────────────
MAIN_PATH = Path(r"C:\Users\legen\Desktop\Lab Project\ESG\v2\단계1_문서선별전처리\ESG_preprocessed_main.jsonl")
ALL_PATH  = Path(r"C:\Users\legen\Desktop\Lab Project\ESG\v2\단계1_문서선별전처리\ESG_preprocessed_all.jsonl")

# main 또는 all 중 선택
INPUT_MODE = "all"   # "main" or "all"

DATA_PATH = MAIN_PATH if INPUT_MODE.lower() == "main" else ALL_PATH

# ── 출력 경로 ──────────────────────────────────────────────────────────────
OUTPUT_DIR = Path(r"C:\Users\legen\Desktop\Lab Project\ESG\v2\단계2_빈도분석")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f"❌ 입력 파일 없음: {DATA_PATH}"
print(f"입력 모드: {INPUT_MODE}")
print(f"입력: {DATA_PATH}")
print(f"출력: {OUTPUT_DIR}")

입력 모드: all
입력: C:\Users\legen\Desktop\Lab Project\ESG\v2\단계1_문서선별전처리\ESG_preprocessed_all.jsonl
출력: C:\Users\legen\Desktop\Lab Project\ESG\v2\단계2_빈도분석


## Cell 3. 공통 함수 정의

- `sent_trigrams_list` 파싱 추가
- Track B strict compact exact 비교를 위한 정규화 함수 추가


In [3]:
def normalize_term(term):
    if term is None:
        return ""
    term = str(term).strip()
    for a, b in [
        ("(", " "), (")", " "), ("[", " "), ("]", " "),
        ("{", " "), ("}", " "), ("·", " "), ("–", "-"),
        ("—", "-"), ("/", " ")
    ]:
        term = term.replace(a, b)
    return re.sub(r"\s+", " ", term).strip()

def normalize_token(token):
    token = normalize_term(token)
    return token.strip("'").strip('"')

def normalize_compact(text):
    """
    Track B strict exact 비교용 정규화.
    - 공백/underscore 제거
    - 영문은 case-insensitive 비교를 위해 lower 처리
    """
    text = normalize_token(text)
    text = text.replace("_", "")
    text = re.sub(r"\s+", "", text)
    return text.lower()

def has_korean(text):
    return bool(re.search(r"[가-힣]", str(text or "")))

def has_alpha(text):
    return bool(re.search(r"[A-Za-z]", str(text or "")))

def is_single_korean_keyword(term):
    term = normalize_term(term)
    return bool(term) and has_korean(term) and (" " not in term) and (not has_alpha(term))

def is_korean_multiword(term):
    term = normalize_term(term)
    return bool(term) and has_korean(term) and (" " in term)

def is_english_or_acronym(term):
    term = normalize_term(term)
    return bool(term) and has_alpha(term) and (not has_korean(term))

def safe_sheet_name(name):
    name = re.sub(r"[\\/*?:\[\]]", "_", str(name))
    return name[:31]

def stringify_categories(cats):
    return "|".join(sorted(set(cats)))

def parse_listlike(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple)):
        return [normalize_token(x) for x in value if normalize_token(x)]
    if pd.isna(value):
        return []

    text = str(value).strip()
    if text in {"", "[]", "{}", "nan", "None", "null"}:
        return []

    for parser in (json.loads, ast.literal_eval):
        try:
            obj = parser(text)
            if isinstance(obj, (list, tuple)):
                return [normalize_token(x) for x in obj if normalize_token(x)]
        except Exception:
            pass

    if re.search(r"[,|;/]", text):
        parts = re.split(r"\s*[,|;/]\s*", text)
        return [normalize_token(x) for x in parts if normalize_token(x)]

    if " " in text and "[" not in text and "]" not in text and "'" not in text and '"' not in text:
        return [normalize_token(x) for x in text.split() if normalize_token(x)]

    return [normalize_token(text)]

def load_input(path):
    p = Path(path)
    assert p.exists(), f"❌ 입력 파일 없음: {p}"
    if p.suffix.lower() == ".jsonl":
        return pd.read_json(p, lines=True)
    elif p.suffix.lower() == ".csv":
        return pd.read_csv(p)
    elif p.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(p)
    else:
        raise ValueError(f"지원하지 않는 파일 형식: {p.suffix}")

def prepare_dataframe(df):
    required = ["text", "nouns_filtered", "sent_bigrams_list"]
    missing = [c for c in required if c not in df.columns]
    assert not missing, f"❌ 필수 컬럼 누락: {missing}"

    df = df.copy()
    df["doc_id"] = np.arange(1, len(df) + 1)
    df["text"] = df["text"].fillna("").astype(str)
    df["char_len"] = df["text"].str.len()
    df["nouns_filtered_list"] = df["nouns_filtered"].apply(parse_listlike)

    if "english_unigrams_filtered" in df.columns:
        df["english_unigrams_filtered_list"] = df["english_unigrams_filtered"].apply(parse_listlike)
    else:
        df["english_unigrams_filtered_list"] = [[] for _ in range(len(df))]

    if "trackA_raw_total_list" in df.columns:
        df["trackA_raw_total_list_parsed"] = df["trackA_raw_total_list"].apply(parse_listlike)
    else:
        df["trackA_raw_total_list_parsed"] = (
            df["nouns_filtered_list"] + df["english_unigrams_filtered_list"]
        )

    df["sent_bigrams_list_parsed"] = df["sent_bigrams_list"].apply(parse_listlike)

    if "sent_trigrams_list" in df.columns:
        df["sent_trigrams_list_parsed"] = df["sent_trigrams_list"].apply(parse_listlike)
    else:
        df["sent_trigrams_list_parsed"] = [[] for _ in range(len(df))]

    df["n_nouns_tokens"] = df["nouns_filtered_list"].apply(len)
    df["n_english_unigram_tokens"] = df["english_unigrams_filtered_list"].apply(len)
    df["n_trackA_raw_tokens"] = df["trackA_raw_total_list_parsed"].apply(len)
    df["n_bigram_tokens"] = df["sent_bigrams_list_parsed"].apply(len)
    df["n_trigram_tokens"] = df["sent_trigrams_list_parsed"].apply(len)

    if "crawl_dual" in df.columns:
        df["page_group_resolved"] = df["crawl_dual"].fillna("unknown").astype(str).str.strip().str.lower()
    elif "page_category" in df.columns:
        df["page_group_resolved"] = df["page_category"].fillna("unknown").astype(str).str.strip().str.lower()
    else:
        df["page_group_resolved"] = "unknown"

    return df

print("공통 함수 정의 완료")


def merge_counter_by_map(counter_obj, alias_to_canonical, canonical_categories=None):
    out = Counter()
    if not isinstance(counter_obj, dict):
        return out
    for term, cnt in counter_obj.items():
        cnt = int(cnt)
        if cnt <= 0:
            continue
        canonical = alias_to_canonical.get(term, term)
        out[canonical] += cnt
    return out

def build_concept_assets():
    ESG_CONCEPT_SPECS = [
        # ESG_META
        {'category': 'ESG_META', 'canonical_ko': 'ESG 평가', 'aliases': ['ESG 평가', 'ESG Rating']},
        {'category': 'ESG_META', 'canonical_ko': '지속가능경영', 'aliases': ['지속가능경영', 'Sustainability Management']},
        {'category': 'ESG_META', 'canonical_ko': '지속가능성 보고서', 'aliases': ['지속가능성 보고서', 'Sustainability Report']},
        {'category': 'ESG_META', 'canonical_ko': '비재무정보', 'aliases': ['비재무정보', 'Non-financial Data']},
        {'category': 'ESG_META', 'canonical_ko': '통합보고서', 'aliases': ['통합보고서', 'Integrated Report']},
        {'category': 'ESG_META', 'canonical_ko': '임팩트 투자', 'aliases': ['임팩트 투자', 'Impact Investing']},
        {'category': 'ESG_META', 'canonical_ko': '책임투자', 'aliases': ['책임투자', 'Responsible Investment', 'RI']},
        {'category': 'ESG_META', 'canonical_ko': '스튜어드십 코드', 'aliases': ['스튜어드십 코드', 'Stewardship Code']},
        {'category': 'ESG_META', 'canonical_ko': '지속가능금융', 'aliases': ['지속가능금융', 'Sustainable Finance']},
        {'category': 'ESG_META', 'canonical_ko': '녹색채권', 'aliases': ['녹색채권', 'Green Bond']},
        {'category': 'ESG_META', 'canonical_ko': '사회적 채권', 'aliases': ['사회적 채권', 'Social Bond']},

        # E
        {'category': 'E', 'canonical_ko': '지속가능', 'aliases': ['지속가능', 'sustainable']},
        {'category': 'E', 'canonical_ko': '지속가능성', 'aliases': ['지속가능성', 'sustainability']},
        {'category': 'E', 'canonical_ko': '그린', 'aliases': ['그린', 'green']},
        {'category': 'E', 'canonical_ko': '기후', 'aliases': ['기후', 'climate']},
        {'category': 'E', 'canonical_ko': '친환경', 'aliases': ['친환경', 'eco-friendly']},
        {'category': 'E', 'canonical_ko': '환경', 'aliases': ['환경', 'environment', 'environmental']},
        {'category': 'E', 'canonical_ko': '탄소', 'aliases': ['탄소', 'carbon']},
        {'category': 'E', 'canonical_ko': '넷제로', 'aliases': ['넷제로', 'net-zero']},
        {'category': 'E', 'canonical_ko': '온실가스', 'aliases': ['온실가스', 'greenhouse gas']},
        {'category': 'E', 'canonical_ko': '가스', 'aliases': ['가스', 'gas']},
        {'category': 'E', 'canonical_ko': '에너지', 'aliases': ['에너지', 'energy']},
        {'category': 'E', 'canonical_ko': '자원', 'aliases': ['자원', 'resource']},
        {'category': 'E', 'canonical_ko': '재활용', 'aliases': ['재활용', 'recycling']},
        {'category': 'E', 'canonical_ko': '업사이클링', 'aliases': ['업사이클링', 'upcycling']},
        {'category': 'E', 'canonical_ko': '재사용', 'aliases': ['재사용', 'reuse']},
        {'category': 'E', 'canonical_ko': '재판매', 'aliases': ['재판매', 'resale']},
        {'category': 'E', 'canonical_ko': '제품수선', 'aliases': ['제품수선', 'repair']},
        {'category': 'E', 'canonical_ko': '오염', 'aliases': ['오염', 'pollution']},
        {'category': 'E', 'canonical_ko': '순환경제', 'aliases': ['순환경제', 'circular economy']},
        {'category': 'E', 'canonical_ko': '동물복지', 'aliases': ['동물복지', 'animal welfare']},
        {'category': 'E', 'canonical_ko': '폐기물', 'aliases': ['폐기물', 'waste']},
        {'category': 'E', 'canonical_ko': '제로 웨이스트', 'aliases': ['제로 웨이스트', 'zero waste']},
        {'category': 'E', 'canonical_ko': '생물다양성', 'aliases': ['생물다양성', 'biodiversity']},
        {'category': 'E', 'canonical_ko': '생태계', 'aliases': ['생태계', 'ecosystems']},
        {'category': 'E', 'canonical_ko': '윤리적', 'aliases': ['윤리적', 'ethical']},

        # S
        {'category': 'S', 'canonical_ko': '사회적 책임', 'aliases': ['사회적 책임', 'social responsibility']},
        {'category': 'S', 'canonical_ko': '책임', 'aliases': ['책임', 'responsibility']},
        {'category': 'S', 'canonical_ko': '책임있는', 'aliases': ['책임있는', 'responsible']},
        {'category': 'S', 'canonical_ko': '노동', 'aliases': ['노동', 'labor']},
        {'category': 'S', 'canonical_ko': '직원', 'aliases': ['직원', 'employee']},
        {'category': 'S', 'canonical_ko': '권리', 'aliases': ['권리', 'rights']},
        {'category': 'S', 'canonical_ko': '복지', 'aliases': ['복지', 'welfare']},
        {'category': 'S', 'canonical_ko': '참여', 'aliases': ['참여', 'engagement']},
        {'category': 'S', 'canonical_ko': '소비자', 'aliases': ['소비자', 'consumer']},
        {'category': 'S', 'canonical_ko': '소비', 'aliases': ['소비', 'consumption']},
        {'category': 'S', 'canonical_ko': '행동', 'aliases': ['행동', 'behavior']},
        {'category': 'S', 'canonical_ko': '보호', 'aliases': ['보호', 'protection']},
        {'category': 'S', 'canonical_ko': '신뢰', 'aliases': ['신뢰', 'trust']},
        {'category': 'S', 'canonical_ko': '인권', 'aliases': ['인권', 'human rights']},
        {'category': 'S', 'canonical_ko': '인적 자본', 'aliases': ['인적 자본', 'human capital']},
        {'category': 'S', 'canonical_ko': '다양성', 'aliases': ['다양성', 'diversity']},
        {'category': 'S', 'canonical_ko': '형평성', 'aliases': ['형평성', 'equity']},
        {'category': 'S', 'canonical_ko': '포용성', 'aliases': ['포용성', 'inclusion']},
        {'category': 'S', 'canonical_ko': '포용적', 'aliases': ['포용적', 'inclusive']},
        {'category': 'S', 'canonical_ko': '지역사회', 'aliases': ['지역사회', 'community']},
        {'category': 'S', 'canonical_ko': '건강', 'aliases': ['건강', 'health']},
        {'category': 'S', 'canonical_ko': '안전', 'aliases': ['안전', 'safety']},
        {'category': 'S', 'canonical_ko': '위생', 'aliases': ['위생', 'sanitation']},
        {'category': 'S', 'canonical_ko': '고객', 'aliases': ['고객', 'customer']},
        {'category': 'S', 'canonical_ko': '빈곤', 'aliases': ['빈곤', 'poverty']},
        {'category': 'S', 'canonical_ko': '기아', 'aliases': ['기아', 'hunger']},
        {'category': 'S', 'canonical_ko': '식량', 'aliases': ['식량', 'food']},
        {'category': 'S', 'canonical_ko': '안보', 'aliases': ['안보', 'security']},
        {'category': 'S', 'canonical_ko': '교육', 'aliases': ['교육', 'education']},
        {'category': 'S', 'canonical_ko': '역량', 'aliases': ['역량', 'upskilling']},
        {'category': 'S', 'canonical_ko': '훈련', 'aliases': ['훈련', 'training']},
        {'category': 'S', 'canonical_ko': '성평등', 'aliases': ['성평등', 'gender equity']},
        {'category': 'S', 'canonical_ko': '여성', 'aliases': ['여성', 'women']},
        {'category': 'S', 'canonical_ko': '성 격차', 'aliases': ['성 격차', 'gender disparities']},
        {'category': 'S', 'canonical_ko': '불평등', 'aliases': ['불평등', 'inequity']},
        {'category': 'S', 'canonical_ko': '공정', 'aliases': ['공정', 'fair']},
        {'category': 'S', 'canonical_ko': '대우', 'aliases': ['대우', 'treatment']},
        {'category': 'S', 'canonical_ko': '임금', 'aliases': ['임금', 'wages']},
        {'category': 'S', 'canonical_ko': '기회', 'aliases': ['기회', 'opportunity']},
        {'category': 'S', 'canonical_ko': '일자리', 'aliases': ['일자리', 'work']},
        {'category': 'S', 'canonical_ko': '경제 성장', 'aliases': ['경제 성장', 'economic growth']},
        {'category': 'S', 'canonical_ko': '비재무', 'aliases': ['비재무', 'non-financial']},
        {'category': 'S', 'canonical_ko': '제품 품질', 'aliases': ['제품 품질', 'product quality']},
        {'category': 'S', 'canonical_ko': '제품수명주기', 'aliases': ['제품수명주기', 'product life cycle']},
        {'category': 'S', 'canonical_ko': '공급망 관리', 'aliases': ['공급망 관리', 'Supply Chain Management']},
        {'category': 'S', 'canonical_ko': '협력사 관리', 'aliases': ['협력사 관리', 'Vendor Management']},
        {'category': 'S', 'canonical_ko': '이해관계자', 'aliases': ['이해관계자', 'Stakeholders']},
        {'category': 'S', 'canonical_ko': '사회적 가치', 'aliases': ['사회적 가치', 'Social Value']},
        {'category': 'S', 'canonical_ko': '웰빙', 'aliases': ['웰빙', 'well-being']},
        {'category': 'S', 'canonical_ko': '개인정보 보호', 'aliases': ['개인정보 보호', 'data privacy']},
        {'category': 'S', 'canonical_ko': '디지털 시민성', 'aliases': ['디지털 시민성', 'digital citizenship']},

        # G
        {'category': 'G', 'canonical_ko': '기업 가치', 'aliases': ['기업 가치', 'firm value']},
        {'category': 'G', 'canonical_ko': '주주 가치', 'aliases': ['주주 가치', 'shareholder value']},
        {'category': 'G', 'canonical_ko': '투명성', 'aliases': ['투명성', 'transparency']},
        {'category': 'G', 'canonical_ko': '공시', 'aliases': ['공시', 'disclosure']},
        {'category': 'G', 'canonical_ko': '윤리', 'aliases': ['윤리', 'ethic']},
        {'category': 'G', 'canonical_ko': '반부패', 'aliases': ['반부패', 'anti-corruption']},
        {'category': 'G', 'canonical_ko': '법규 준수', 'aliases': ['법규 준수', 'legal compliance']},
        {'category': 'G', 'canonical_ko': '이사회', 'aliases': ['이사회', 'Board of Directors']},
        {'category': 'G', 'canonical_ko': '독립성', 'aliases': ['독립성', 'Board Independence']},
        {'category': 'G', 'canonical_ko': '감사위원회', 'aliases': ['감사위원회', 'Audit Committee']},
        {'category': 'G', 'canonical_ko': '경영진 보상', 'aliases': ['경영진 보상', 'Executive Compensation']},
        {'category': 'G', 'canonical_ko': '주주권', 'aliases': ['주주권', 'Shareholder Rights']},
        {'category': 'G', 'canonical_ko': '파트너쉽', 'aliases': ['파트너쉽', 'partnership']},
        {'category': 'G', 'canonical_ko': '협력', 'aliases': ['협력', 'collaboration']},
        {'category': 'G', 'canonical_ko': '기업지배구조', 'aliases': ['기업지배구조', 'Corporate Governance']},
    ]

    alias_to_canonical = {}
    canonical_to_categories = defaultdict(set)
    for spec in ESG_CONCEPT_SPECS:
        canonical = normalize_token(spec['canonical_ko'])
        category = spec['category']
        canonical_to_categories[canonical].add(category)
        for alias in spec['aliases']:
            alias_n = normalize_token(alias)
            if alias_n:
                alias_to_canonical[alias_n] = canonical

    return {
        'specs': ESG_CONCEPT_SPECS,
        'alias_to_canonical': alias_to_canonical,
        'canonical_vocab': sorted(canonical_to_categories.keys()),
        'canonical_categories': {k: sorted(v) for k, v in canonical_to_categories.items()},
    }


공통 함수 정의 완료


## Cell 4. ESG 키워드 · Track 규칙 정의

- Track A: 단일 한국어 키워드 exact
- Track B: 현재 ESG 키워드 전체를 대상으로 compact exact(공백/underscore 제거 후 exact 비교)
- Track B partial: bigram/trigram 내부 단일 토큰이 Track A와 일치할 때 별도 저장
- Track C: 영문 약어 regex


In [4]:
# ══════════════════════════════════════════════════════════════════════════
# 4-1. ESG 키워드 사전
# ══════════════════════════════════════════════════════════════════════════
RAW_KEYWORDS_BY_CATEGORY = {
    "ESG_META": [
        "ESG 평가", "ESG Rating", "지속가능경영", "Sustainability Management",
        "지속가능성 보고서", "Sustainability Report", "비재무정보", "Non-financial Data",
        "통합보고서", "Integrated Report", "임팩트 투자", "Impact Investing",
        "책임투자", "Responsible Investment", "RI", "스튜어드십 코드", "Stewardship Code",
        "TCFD", "SASB", "GRI기준", "ISSB 기준", "ESG KPI", "ESG지표",
        "지속가능금융", "Sustainable Finance", "녹색채권", "Green Bond",
        "사회적 채권", "Social Bond",
    ],
    "E": [
        "지속가능", "sustainable", "지속가능성", "sustainability",
        "그린", "green", "기후", "climate", "친환경", "eco-friendly",
        "환경", "environment", "environmental", "탄소", "carbon",
        "넷제로", "net-zero", "온실가스", "greenhouse gas", "가스", "gas", "GHG",
        "Scope 1", "Scope 2", "Scope 3 배출", "에너지", "energy",
        "태양광", "풍력", "대기전력", "자원", "resource",
        "재활용", "recycling", "업사이클링", "upcycling", "분리배출",
        "재사용", "reuse", "재판매", "resale", "제품수선", "repair",
        "오염", "pollution", "미세먼지", "플라스틱",
        "순환경제", "circular economy", "동물복지", "animal welfare",
        "물", "water", "폐기물", "waste", "제로 웨이스트", "zero waste",
        "생물다양성", "biodiversity", "생태계", "ecosystems", "윤리적", "ethical",
        "화학물질", "유해물질", "절약", "감축", "관리", "대체", "행동",
        "복지", "실천", "보호", "보전", "감시",
    ],
    "S": [
        "사회적 책임", "social responsibility", "책임", "responsibility", "책임있는", "responsible",
        "노동", "labor", "직원", "employee", "권리", "rights", "복지", "welfare",
        "참여", "engagement", "소비자", "consumer", "소비", "consumption",
        "행동", "behavior", "보호", "protection", "신뢰", "trust", "인권", "human rights",
        "인적 자본", "human capital", "다양성", "diversity", "평등", "형평성", "equity",
        "포용성", "inclusion", "포용적", "inclusive", "지역사회", "community",
        "개발", "활동", "의식", "공헌", "건강", "health", "안전", "safety",
        "위생", "sanitation", "고객", "customer", "빈곤", "poverty",
        "기아", "hunger", "식량", "food", "안보", "security",
        "교육", "education", "역량", "upskilling", "훈련", "training",
        "성평등", "gender equity", "여성", "women", "성 격차", "gender disparities",
        "불평등", "inequity", "공정", "fair", "대우", "treatment", "임금", "wages",
        "기회", "opportunity", "일자리", "work", "경제 성장", "economic growth",
        "비재무", "non-financial", "제품 품질", "product quality",
        "제품수명주기", "product life cycle", "공급망 관리", "Supply Chain Management",
        "협력사 관리", "Vendor Management", "이해관계자", "Stakeholders",
        "사회적 가치", "Social Value", "관계", "돌봄", "웰빙", "well-being",
        "삶의 질", "윤리", "ethic", "개인정보 보호", "data privacy", "디지털 시민성", "digital citizenship",
    ],
    "G": [
        "ESG 환경경영", "ESG 위원회", "기업 가치", "firm value", "주주 가치", "shareholder value",
        "투명성", "transparency", "공시", "disclosure", "표시제도", "윤리", "ethical",
        "반부패", "anti-corruption", "뇌물방지", "법규 준수", "legal compliance",
        "이사회", "Board of Directors", "독립성", "Board Independence",
        "감사위원회", "Audit Committee", "경영진 보상", "Executive Compensation",
        "주주권", "Shareholder Rights", "파트너쉽", "partnership",
        "협력", "collaboration", "기업지배구조", "Corporate Governance",
        "가계관리", "가계재무", "소비계획", "의사결정 기준",
        "법", "규칙", "책임", "정보 활용", "정보 판단",
    ],
}

# false positive 전면 제외
EXCLUDE_KEYWORDS = {"법", "물"}

# 복수 카테고리 동시 집계
KW_MULTI_CAT = {
    "복지": ["E", "S"],
    "관리": ["E", "S"],
    "보호": ["E", "S"],
    "행동": ["E", "S"],
    "윤리": ["S", "G"],
    "협력": ["G", "S"],
    "실천": ["E", "S"],
}

# Track C: 영문 약어 regex
TRACK_C_REGEX_PATTERNS = {
    "ESG":  (r"(?<![A-Za-z])ESG(?![A-Za-z])", ["ESG_META"]),
    "GRI":  (r"(?<![A-Za-z])GRI(?![A-Za-z])", ["ESG_META"]),
    "TCFD": (r"(?<![A-Za-z])TCFD(?![A-Za-z])", ["ESG_META"]),
    "SASB": (r"(?<![A-Za-z])SASB(?![A-Za-z])", ["ESG_META"]),
    "GHG":  (r"(?<![A-Za-z])GHG(?![A-Za-z])", ["E"]),
    "ISSB": (r"(?<![A-Za-z])ISSB(?![A-Za-z])", ["ESG_META"]),
}

def build_keyword_assets():
    kw_to_cats = defaultdict(set)
    for cat, terms in RAW_KEYWORDS_BY_CATEGORY.items():
        for term in terms:
            term_n = normalize_term(term)
            if term_n:
                kw_to_cats[term_n].add(cat)

    for kw, cats in KW_MULTI_CAT.items():
        kw_to_cats[normalize_term(kw)].update(cats)

    track_a_vocab = []
    track_a_categories = {}
    track_b_compact_lookup = {}
    track_b_categories = {}
    dropped_multis_for_track_a = []
    dropped_english_for_track_a = []
    excluded_keywords = []

    for kw, cats_set in sorted(kw_to_cats.items()):
        if not kw:
            continue
        if kw in EXCLUDE_KEYWORDS:
            excluded_keywords.append(kw)
            continue

        cats = sorted(cats_set)

        # Track A: 단일 한국어
        if is_single_korean_keyword(kw):
            track_a_vocab.append(kw)
            track_a_categories[kw] = cats
        elif is_korean_multiword(kw):
            dropped_multis_for_track_a.append(kw)
        elif is_english_or_acronym(kw):
            dropped_english_for_track_a.append(kw)
        else:
            if has_alpha(kw):
                dropped_english_for_track_a.append(kw)

        # Track B strict target:
        # 현재 ESG 키워드 전체를 대상으로 compact exact 비교(공백/underscore 제거)
        compact = normalize_compact(kw)
        if compact:
            if compact not in track_b_compact_lookup:
                track_b_compact_lookup[compact] = kw
            track_b_categories[track_b_compact_lookup[compact]] = sorted(
                set(track_b_categories.get(track_b_compact_lookup[compact], [])) | set(cats)
            )

    assets = {
        "keyword_to_categories": {k: sorted(v) for k, v in kw_to_cats.items() if k not in EXCLUDE_KEYWORDS},
        "track_a_vocab": sorted(set(track_a_vocab)),
        "track_a_categories": track_a_categories,
        "track_b_compact_lookup": track_b_compact_lookup,    # compact -> display keyword
        "track_b_canonical": sorted(track_b_categories.keys()),  # display keyword list
        "track_b_categories": track_b_categories,
        "track_c_patterns": {k: re.compile(v[0], flags=re.IGNORECASE) for k, v in TRACK_C_REGEX_PATTERNS.items()},
        "track_c_categories": {k: v[1] for k, v in TRACK_C_REGEX_PATTERNS.items()},
        "dropped_multis_for_track_a": sorted(set(dropped_multis_for_track_a)),
        "dropped_english_for_track_a": sorted(set(dropped_english_for_track_a)),
        "excluded_keywords": sorted(set(excluded_keywords)),
    }
    return assets

print("키워드 · Track 규칙 정의 완료")


키워드 · Track 규칙 정의 완료


## Cell 5. 입력 데이터 로드

In [5]:
df_raw = load_input(DATA_PATH)
df = prepare_dataframe(df_raw)

print(f"로드 완료: {len(df):,}건")
print(f"컬럼 수: {len(df.columns)}개")
print(f"컬럼명: {list(df.columns)}")

display_cols = [c for c in ["crawl_dual", "page_category", "page_type", "대학명", "계열"] if c in df.columns]
for col in display_cols:
    print(f"\n[{col}] 분포")
    print(df[col].value_counts(dropna=False).head(20))

로드 완료: 1,054건
컬럼 수: 42개
컬럼명: ['대학명', '단과대학', '계열', '메뉴명', 'page_type', 'page_type_label', 'page_category', 'crawl_dual', 'source_url', 'collected_url', 'domain', '설립구분', '성별구분', 'text', 'text_length', 'text_len_bin', 'is_esg', 'has_ESG_총괄', 'has_E', 'has_S', 'has_G', 'nouns_list', 'nouns_filtered', 'english_unigrams_list', 'english_unigrams_filtered', 'trackA_raw_total_list', 'sent_bigrams_list', 'sent_trigrams_list', '_text_hash', 'doc_id', 'char_len', 'nouns_filtered_list', 'english_unigrams_filtered_list', 'trackA_raw_total_list_parsed', 'sent_bigrams_list_parsed', 'sent_trigrams_list_parsed', 'n_nouns_tokens', 'n_english_unigram_tokens', 'n_trackA_raw_tokens', 'n_bigram_tokens', 'n_trigram_tokens', 'page_group_resolved']

[crawl_dual] 분포
crawl_dual
main     866
promo    188
Name: count, dtype: int64

[page_category] 분포
page_category
main     866
promo    188
Name: count, dtype: int64

[page_type] 분포
page_type
ㄹ    362
ㄱ    180
ㅋ    152
ㄷ    137
ㅈ     70
ㅁ     48
ㅂ     47
ㅌ     19
ㅅ

## Cell 6. 사전 점검 및 Track 자산 구성

- `infer_track_b_assets_from_df`는 사용하지 않음
- Track B는 현재 ESG 키워드 전체를 compact exact 방식으로 구성


In [6]:
assets = build_keyword_assets()
concept_assets = build_concept_assets()

qa_rows = [
    ("n_docs", len(df)),
    ("n_docs_nonempty_text", int((df["char_len"] > 0).sum())),
    ("n_docs_nouns_nonempty", int((df["n_nouns_tokens"] > 0).sum())),
    ("n_docs_bigram_nonempty", int((df["n_bigram_tokens"] > 0).sum())),
    ("n_docs_trigram_nonempty", int((df["n_trigram_tokens"] > 0).sum())),
    ("n_docs_english_unigram_nonempty", int((df["n_english_unigram_tokens"] > 0).sum())),
    ("n_docs_trackA_raw_nonempty", int((df["n_trackA_raw_tokens"] > 0).sum())),
    ("track_a_vocab_size", len(assets["track_a_vocab"])),
    ("track_a_concept_vocab_size", len(concept_assets["canonical_vocab"])),
    ("track_b_canonical_size", len(assets["track_b_canonical"])),
    ("track_c_pattern_size", len(assets["track_c_patterns"])),
    ("excluded_keywords_n", len(assets["excluded_keywords"])),
    ("dropped_multis_for_track_a_n", len(assets["dropped_multis_for_track_a"])),
    ("dropped_english_for_track_a_n", len(assets["dropped_english_for_track_a"])),
]
qa_preflight = pd.DataFrame(qa_rows, columns=["item", "value"])
print(qa_preflight.to_string(index=False))

print("\n[Track A vocab sample]")
print(assets["track_a_vocab"][:30])

print("\n[Track B canonical sample]")
print(assets["track_b_canonical"][:30])

print("\n[Track C labels]")
print(list(assets["track_c_patterns"].keys()))


                           item  value
                         n_docs   1054
           n_docs_nonempty_text   1054
          n_docs_nouns_nonempty   1052
         n_docs_bigram_nonempty   1051
        n_docs_trigram_nonempty   1051
n_docs_english_unigram_nonempty    411
     n_docs_trackA_raw_nonempty   1053
             track_a_vocab_size    110
     track_a_concept_vocab_size    102
         track_b_canonical_size    251
           track_c_pattern_size      6
            excluded_keywords_n      2
   dropped_multis_for_track_a_n     28
  dropped_english_for_track_a_n    113

[Track A vocab sample]
['가계관리', '가계재무', '가스', '감사위원회', '감시', '감축', '개발', '건강', '고객', '공시', '공정', '공헌', '관계', '관리', '교육', '권리', '규칙', '그린', '기아', '기업지배구조', '기회', '기후', '넷제로', '노동', '녹색채권', '뇌물방지', '다양성', '대기전력', '대우', '대체']

[Track B canonical sample]
['Audit Committee', 'Board Independence', 'Board of Directors', 'Corporate Governance', 'ESG KPI', 'ESG Rating', 'ESG 위원회', 'ESG 평가', 'ESG 환경경영', 'ESG지표', 'Executi

## Cell 7. 내부 로직 self-check

- Track B exact: bigram + trigram 대상 compact exact
- Track B partial: bigram/trigram 내부 토큰 중 Track A 일치분 별도 저장


In [7]:
def analyze_document(text, nouns_tokens, english_unigram_tokens, track_a_raw_tokens, bigram_tokens, trigram_tokens, assets, concept_assets):
    track_a = Counter()
    track_a_concept = Counter()
    track_b_exact = Counter()
    track_b_exact_concept = Counter()
    track_b_partial = Counter()
    track_c = Counter()
    track_c_concept = Counter()

    track_a_set = set(assets["track_a_vocab"])

    for tok in track_a_raw_tokens:
        tok_n = normalize_token(tok)
        if tok_n in track_a_set or tok_n in concept_assets["alias_to_canonical"]:
            track_a[tok_n] += 1
            canonical = concept_assets["alias_to_canonical"].get(tok_n)
            if canonical:
                track_a_concept[canonical] += 1

    # Track B exact: bigram + trigram, compact exact
    for raw_ng in list(bigram_tokens) + list(trigram_tokens):
        ng = normalize_token(raw_ng).replace(" ", "_")
        ng = re.sub(r"_+", "_", ng)
        if not ng:
            continue

        compact = normalize_compact(ng)
        if compact in assets["track_b_compact_lookup"]:
            display_kw = assets["track_b_compact_lookup"][compact]
            track_b_exact[display_kw] += 1
            canonical = concept_assets["alias_to_canonical"].get(display_kw)
            if canonical:
                track_b_exact_concept[canonical] += 1

        # partial은 bigram/trigram 내부 토큰 중 Track A exact만 별도 저장
        parts = [p for p in ng.split("_") if p]
        for part in parts:
            if part in track_a_set:
                track_b_partial[part] += 1

    for label, pattern in assets["track_c_patterns"].items():
        matches = pattern.findall(str(text or ""))
        if matches:
            track_c[label] += len(matches)

    track_c_concept_map = {
        "ESG": "ESG 평가",
        "GRI": "GRI기준",
        "TCFD": "TCFD",
        "SASB": "SASB",
        "GHG": "GHG",
        "ISSB": "ISSB 기준",
    }
    for label, cnt in track_c.items():
        canonical = track_c_concept_map.get(label)
        if canonical:
            track_c_concept[canonical] += int(cnt)

    return {
        "track_a": track_a,
        "track_a_concept": track_a_concept,
        "track_b_exact": track_b_exact,
        "track_b_exact_concept": track_b_exact_concept,
        "track_b_partial": track_b_partial,
        "track_c": track_c,
        "track_c_concept": track_c_concept,
    }

def run_synthetic_self_check():
    test_df = pd.DataFrame({
        "text": [
            "ESG GRI TCFD 관련 문서. 사회적 책임과 기후 위기, 제품 수선을 다룬다. GHG도 언급.",
            "ISSB 기준과 ESG 평가. 개인정보 보호, 성 평등, 비재무정보를 다룬다.",
            "일반 문서. 키워드 거의 없음."
        ],
        "nouns_filtered": [
            '["기후", "책임", "지속가능성", "친환경"]',
            '["안전", "보호", "소비자", "평등"]',
            '[]'
        ],
        "sent_bigrams_list": [
            '["기후_위기", "제품_수선"]',
            '["개인정보_보호", "성_평등", "ESG_평가"]',
            '[]'
        ],
        "sent_trigrams_list": [
            '["사회_적_책임"]',
            '["비_재무_정보", "ISSB_기준"]',
            '[]'
        ],
        "english_unigrams_filtered": [
            '["GHG"]',
            '["education"]',
            '[]'
        ],
        "trackA_raw_total_list": [
            '["기후", "책임", "지속가능성", "친환경", "GHG"]',
            '["안전", "보호", "소비자", "평등", "education"]',
            '[]'
        ],
        "crawl_dual": ["main", "promo", "main"]
    })

    test_df = prepare_dataframe(test_df)
    test_assets = build_keyword_assets()
    test_concept_assets = build_concept_assets()

    results = []
    for _, row in test_df.iterrows():
        results.append(analyze_document(
            text=row["text"],
            nouns_tokens=row["nouns_filtered_list"],
            english_unigram_tokens=row["english_unigrams_filtered_list"],
            track_a_raw_tokens=row["trackA_raw_total_list_parsed"],
            bigram_tokens=row["sent_bigrams_list_parsed"],
            trigram_tokens=row["sent_trigrams_list_parsed"],
            assets=test_assets,
            concept_assets=test_concept_assets
        ))

    test_df["track_a_counter"] = [x["track_a"] for x in results]
    test_df["track_a_concept_counter"] = [x["track_a_concept"] for x in results]
    test_df["track_b_exact_counter"] = [x["track_b_exact"] for x in results]
    test_df["track_b_exact_concept_counter"] = [x["track_b_exact_concept"] for x in results]
    test_df["track_b_partial_counter"] = [x["track_b_partial"] for x in results]
    test_df["track_c_counter"] = [x["track_c"] for x in results]
    test_df["track_c_concept_counter"] = [x["track_c_concept"] for x in results]

    assert test_df.loc[0, "track_a_counter"]["친환경"] == 1, "Track A exact 실패"
    assert test_df.loc[1, "track_a_counter"]["education"] == 1, "Track A 영문 raw 실패"
    assert test_df.loc[1, "track_a_concept_counter"]["교육"] == 1, "Track A concept 병합 실패"
    assert test_df.loc[0, "track_b_exact_counter"]["사회적 책임"] == 1, "Track B trigram compact exact 실패"
    assert test_df.loc[0, "track_b_exact_counter"]["제품수선"] == 1, "Track B 결합형 복원 실패"
    assert test_df.loc[1, "track_b_exact_counter"]["개인정보 보호"] == 1, "Track B 공백형 복원 실패"
    assert test_df.loc[1, "track_b_exact_counter"]["ESG 평가"] == 1, "Track B 혼합형 공백 복원 실패"
    assert test_df.loc[1, "track_b_exact_counter"]["성평등"] == 1, "Track B 무공백 결합형 복원 실패"
    assert test_df.loc[1, "track_b_exact_counter"]["비재무정보"] == 1, "Track B 3토큰 복원 실패"
    assert test_df.loc[1, "track_b_exact_counter"]["ISSB 기준"] == 1, "Track B 혼합형 exact 실패"
    assert test_df.loc[0, "track_b_exact_concept_counter"]["사회적 책임"] == 1, "Track B concept 병합 실패"
    assert test_df.loc[0, "track_b_partial_counter"]["기후"] == 1, "Track B partial 실패"
    assert test_df.loc[0, "track_c_counter"]["ESG"] == 1, "Track C ESG regex 실패"
    assert test_df.loc[0, "track_c_counter"]["GRI"] == 1, "Track C GRI regex 실패"
    assert test_df.loc[0, "track_c_counter"]["TCFD"] == 1, "Track C TCFD regex 실패"
    assert test_df.loc[0, "track_c_counter"]["GHG"] == 1, "Track C GHG regex 실패"
    assert test_df.loc[1, "track_c_counter"]["ISSB"] == 1, "Track C ISSB regex 실패"
    assert test_df.loc[0, "track_c_concept_counter"]["ESG 평가"] == 1, "Track C concept ESG 실패"

    docs = test_df[test_df["n_trackA_raw_tokens"] > 0]["trackA_raw_total_list_parsed"].apply(lambda x: " ".join(x)).tolist()
    vec = TfidfVectorizer(tokenizer=str.split, preprocessor=None, token_pattern=None, lowercase=False, vocabulary=sorted(set(test_assets["track_a_vocab"]) | set(test_concept_assets["alias_to_canonical"].keys())))
    X = vec.fit_transform(docs)
    tfidf_cols = list(vec.get_feature_names_out())
    assert "친환경" in tfidf_cols, "TF-IDF 친환경 누락"

    print("[SELF-CHECK PASS]")
    print(f"- Track A vocab size: {len(test_assets['track_a_vocab'])}")
    print(f"- Track B canonical size: {len(test_assets['track_b_canonical'])}")
    print(f"- Track C regex size: {len(test_assets['track_c_patterns'])}")

run_synthetic_self_check()


[SELF-CHECK PASS]
- Track A vocab size: 110
- Track B canonical size: 251
- Track C regex size: 6


## Cell 8. 문서별 Track A/B/C 분석 실행

In [8]:
results = []
for _, row in df.iterrows():
    results.append(analyze_document(
        text=row["text"],
        nouns_tokens=row["nouns_filtered_list"],
        english_unigram_tokens=row["english_unigrams_filtered_list"],
        track_a_raw_tokens=row["trackA_raw_total_list_parsed"],
        bigram_tokens=row["sent_bigrams_list_parsed"],
        trigram_tokens=row["sent_trigrams_list_parsed"],
        assets=assets,
        concept_assets=concept_assets
    ))

df["track_a_counter"] = [x["track_a"] for x in results]
df["track_a_concept_counter"] = [x["track_a_concept"] for x in results]
df["track_b_exact_counter"] = [x["track_b_exact"] for x in results]
df["track_b_exact_concept_counter"] = [x["track_b_exact_concept"] for x in results]
df["track_b_partial_counter"] = [x["track_b_partial"] for x in results]
df["track_c_counter"] = [x["track_c"] for x in results]
df["track_c_concept_counter"] = [x["track_c_concept"] for x in results]

df["trackA_hits_total"] = df["track_a_counter"].apply(lambda c: int(sum(c.values())))
df["trackB_exact_hits_total"] = df["track_b_exact_counter"].apply(lambda c: int(sum(c.values())))
df["trackB_partial_hits_total"] = df["track_b_partial_counter"].apply(lambda c: int(sum(c.values())))
df["trackC_hits_total"] = df["track_c_counter"].apply(lambda c: int(sum(c.values())))
df["trackABC_hits_total"] = df["trackA_hits_total"] + df["trackB_exact_hits_total"] + df["trackC_hits_total"]

print("문서별 Track 분석 완료")
print(df[["doc_id", "trackA_hits_total", "trackB_exact_hits_total", "trackB_partial_hits_total", "trackC_hits_total"]].head())


문서별 Track 분석 완료
   doc_id  trackA_hits_total  trackB_exact_hits_total  \
0       1                  7                        0   
1       2                 37                        0   
2       3                 14                        0   
3       4                 42                        0   
4       5                144                        5   

   trackB_partial_hits_total  trackC_hits_total  
0                         35                  0  
1                        165                  0  
2                         57                  0  
3                        196                  0  
4                        678                  0  


## Cell 9. 카운트 매트릭스 · TF-IDF · 전체 요약표 생성

In [9]:
def build_counter_matrix(df, counter_col, all_terms):
    rows, idx = [], []
    for _, row in df.iterrows():
        counter = row[counter_col]
        rows.append({term: int(counter.get(term, 0)) for term in all_terms})
        idx.append(row["doc_id"])
    mat = pd.DataFrame(rows, index=idx)
    mat.index.name = "doc_id"
    return mat

def compute_tfidf_matrix(df, vocab, token_col="trackA_raw_total_list_parsed", min_token_col="n_trackA_raw_tokens"):
    subset = df[df[min_token_col] > 0].copy()
    if subset.empty or not vocab:
        return pd.DataFrame(index=pd.Index([], name="doc_id"))
    docs = subset[token_col].apply(lambda x: " ".join(x)).tolist()
    vec = TfidfVectorizer(
        tokenizer=str.split,
        preprocessor=None,
        token_pattern=None,
        lowercase=False,
        vocabulary=list(vocab)
    )
    X = vec.fit_transform(docs)
    out = pd.DataFrame.sparse.from_spmatrix(
        X,
        index=subset["doc_id"],
        columns=vec.get_feature_names_out()
    )
    out.index.name = "doc_id"
    return out

# Track B exact / Track C처럼 '매칭된 키워드 카운터'에 기반한 TF-IDF 계산
def compute_tfidf_from_counter(df, counter_col, vocab):
    """
    counter_col에 저장된 Counter(dict)를 기반으로 TF-IDF 계산
    - Track B exact / Track C처럼 '매칭된 키워드 카운터'에 사용
    """
    if df.empty or not vocab:
        return pd.DataFrame(index=pd.Index([], name="doc_id"))

    docs = []
    idx = []

    for _, row in df.iterrows():
        ctr = row[counter_col]
        if not isinstance(ctr, dict):
            ctr = {}

        terms = []
        for term in vocab:
            n = int(ctr.get(term, 0))
            if n > 0:
                terms.extend([term] * n)

        # 빈 문서가 있어도 vectorizer가 깨지지 않게 placeholder 처리
        if not terms:
            terms = ["__EMPTY__"]

        docs.append(" ".join(terms))
        idx.append(row["doc_id"])

    vec = TfidfVectorizer(
        tokenizer=str.split,
        preprocessor=None,
        token_pattern=None,
        lowercase=False,
        vocabulary=list(vocab)
    )
    X = vec.fit_transform(docs)

    out = pd.DataFrame.sparse.from_spmatrix(
        X,
        index=idx,
        columns=vec.get_feature_names_out()
    )
    out.index.name = "doc_id"

    return out


def term_summary_from_matrix(count_matrix, terms, categories_map, main_doc_ids):
    if count_matrix.empty:
        return pd.DataFrame(columns=[
            "keyword", "categories",
            "raw_count_overall", "doc_freq_overall",
            "raw_count_main", "doc_freq_main"
        ])

    overall_sum = count_matrix.sum(axis=0)
    overall_df = (count_matrix > 0).sum(axis=0)
    main_matrix = count_matrix.loc[count_matrix.index.intersection(list(main_doc_ids))]

    if main_matrix.empty:
        main_sum = pd.Series(0, index=count_matrix.columns)
        main_df = pd.Series(0, index=count_matrix.columns)
    else:
        main_sum = main_matrix.sum(axis=0)
        main_df = (main_matrix > 0).sum(axis=0)

    out = pd.DataFrame({
        "keyword": list(terms),
        "categories": [stringify_categories(categories_map.get(t, [])) for t in terms],
        "raw_count_overall": [int(overall_sum.get(t, 0)) for t in terms],
        "doc_freq_overall": [int(overall_df.get(t, 0)) for t in terms],
        "raw_count_main": [int(main_sum.get(t, 0)) for t in terms],
        "doc_freq_main": [int(main_df.get(t, 0)) for t in terms],
    })
    return out.sort_values(
        by=["raw_count_overall", "doc_freq_overall", "keyword"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

def attach_tfidf_metrics(track_a_summary, tfidf_matrix, main_doc_ids):
    out = track_a_summary.copy()
    if tfidf_matrix.empty:
        out["mean_tfidf_overall"] = 0.0
        out["mean_tfidf_main"] = 0.0
        return out

    overall_mean = tfidf_matrix.mean(axis=0)
    main_idx = tfidf_matrix.index.intersection(list(main_doc_ids))
    if len(main_idx):
        main_mean = tfidf_matrix.loc[main_idx].mean(axis=0)
    else:
        main_mean = pd.Series(0, index=tfidf_matrix.columns)

    out["mean_tfidf_overall"] = out["keyword"].map(lambda x: float(overall_mean.get(x, 0.0)))
    out["mean_tfidf_main"] = out["keyword"].map(lambda x: float(main_mean.get(x, 0.0)))

    return out.sort_values(
        by=["raw_count_overall", "doc_freq_overall", "mean_tfidf_overall"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

def category_summary_from_term_summary(term_summary, track_name):
    records = []
    for _, row in term_summary.iterrows():
        cats = [c for c in str(row["categories"]).split("|") if c]
        for cat in cats:
            records.append({
                "track": track_name,
                "category": cat,
                "raw_count_overall": row["raw_count_overall"],
                "raw_count_main": row["raw_count_main"],
            })
    if not records:
        return pd.DataFrame(columns=["track", "category", "raw_count_overall", "raw_count_main"])

    out = pd.DataFrame(records)
    out = out.groupby(["track", "category"], as_index=False)[["raw_count_overall", "raw_count_main"]].sum()
    return out.sort_values(by=["raw_count_overall", "raw_count_main"], ascending=False).reset_index(drop=True)

main_doc_ids = set(df.loc[df["page_group_resolved"].eq("main"), "doc_id"].tolist())
if not main_doc_ids:
    main_doc_ids = set(df["doc_id"].tolist())

track_a_vocab_raw = sorted(set(assets["track_a_vocab"]) | set(df["english_unigrams_filtered_list"].explode().dropna().astype(str).map(normalize_token).tolist()))
track_a_categories_raw = dict(assets["track_a_categories"])
for alias, canonical in concept_assets["alias_to_canonical"].items():
    if alias in track_a_vocab_raw and canonical in concept_assets["canonical_categories"]:
        track_a_categories_raw[alias] = concept_assets["canonical_categories"][canonical]

track_a_mat = build_counter_matrix(df, "track_a_counter", track_a_vocab_raw)
track_a_concept_mat = build_counter_matrix(df, "track_a_concept_counter", concept_assets["canonical_vocab"])
track_b_exact_mat = build_counter_matrix(df, "track_b_exact_counter", assets["track_b_canonical"])
track_b_exact_concept_mat = build_counter_matrix(df, "track_b_exact_concept_counter", concept_assets["canonical_vocab"])
track_b_partial_mat = build_counter_matrix(df, "track_b_partial_counter", assets["track_a_vocab"])
track_c_mat = build_counter_matrix(df, "track_c_counter", list(assets["track_c_patterns"].keys()))
track_c_concept_vocab = ["ESG 평가", "GRI기준", "TCFD", "SASB", "GHG", "ISSB 기준"]
track_c_concept_categories = {
    "ESG 평가": ["ESG_META"], "GRI기준": ["ESG_META"], "TCFD": ["ESG_META"],
    "SASB": ["ESG_META"], "GHG": ["E"], "ISSB 기준": ["ESG_META"]
}
track_c_concept_mat = build_counter_matrix(df, "track_c_concept_counter", track_c_concept_vocab)

tfidf_trackA = compute_tfidf_matrix(df, track_a_vocab_raw, token_col="trackA_raw_total_list_parsed", min_token_col="n_trackA_raw_tokens")
tfidf_trackA_concept = compute_tfidf_from_counter(df, "track_a_concept_counter", concept_assets["canonical_vocab"])
tfidf_trackB = compute_tfidf_from_counter(df, "track_b_exact_counter", assets["track_b_canonical"])
tfidf_trackB_concept = compute_tfidf_from_counter(df, "track_b_exact_concept_counter", concept_assets["canonical_vocab"])
tfidf_trackC = compute_tfidf_from_counter(df, "track_c_counter", list(assets["track_c_patterns"].keys()))
tfidf_trackC_concept = compute_tfidf_from_counter(df, "track_c_concept_counter", track_c_concept_vocab)

#tfidf_mat = compute_tfidf_matrix(df, assets["track_a_vocab"])

trackA_overall = attach_tfidf_metrics(
    term_summary_from_matrix(track_a_mat, track_a_vocab_raw, track_a_categories_raw, main_doc_ids),
    tfidf_trackA,
    main_doc_ids,
)

trackA_concept_overall = attach_tfidf_metrics(
    term_summary_from_matrix(track_a_concept_mat, concept_assets["canonical_vocab"], concept_assets["canonical_categories"], main_doc_ids),
    tfidf_trackA_concept,
    main_doc_ids,
)

trackB_exact_overall = attach_tfidf_metrics(
    term_summary_from_matrix(track_b_exact_mat, assets["track_b_canonical"], assets["track_b_categories"], main_doc_ids),
    tfidf_trackB,
    main_doc_ids,
)

trackB_exact_concept_overall = attach_tfidf_metrics(
    term_summary_from_matrix(track_b_exact_concept_mat, concept_assets["canonical_vocab"], concept_assets["canonical_categories"], main_doc_ids),
    tfidf_trackB_concept,
    main_doc_ids,
)

trackB_partial_overall = term_summary_from_matrix(
    track_b_partial_mat,
    assets["track_a_vocab"],
    assets["track_a_categories"],
    main_doc_ids
)

trackC_overall = attach_tfidf_metrics(
    term_summary_from_matrix(track_c_mat, list(assets["track_c_patterns"].keys()), assets["track_c_categories"], main_doc_ids),
    tfidf_trackC,
    main_doc_ids,
)

trackC_concept_overall = attach_tfidf_metrics(
    term_summary_from_matrix(track_c_concept_mat, track_c_concept_vocab, track_c_concept_categories, main_doc_ids),
    tfidf_trackC_concept,
    main_doc_ids,
)

trackA_category_summary = category_summary_from_term_summary(trackA_overall, "A_raw")
trackA_concept_category_summary = category_summary_from_term_summary(trackA_concept_overall, "A_concept")
trackB_exact_category_summary = category_summary_from_term_summary(trackB_exact_overall, "B_exact_raw")
trackB_exact_concept_category_summary = category_summary_from_term_summary(trackB_exact_concept_overall, "B_exact_concept")
trackB_partial_category_summary = category_summary_from_term_summary(trackB_partial_overall, "B_partial")
trackC_category_summary = category_summary_from_term_summary(trackC_overall, "C_raw")
trackC_concept_category_summary = category_summary_from_term_summary(trackC_concept_overall, "C_concept")

print("[Track A overall 상위 20]")
print(trackA_overall.head(20).to_string(index=False))

print("\n[Track B exact overall 상위 20]")
print(trackB_exact_overall.head(20).to_string(index=False))

print("\n[Track C overall]")
print(trackC_overall.to_string(index=False))

[Track A overall 상위 20]
  keyword categories  raw_count_overall  doc_freq_overall  raw_count_main  doc_freq_main  mean_tfidf_overall  mean_tfidf_main
       교육          S               3539               557            2730            466            0.227011         0.210811
      소비자          S               3167               223            3020            207            0.106949         0.116264
       개발          S               1808               435            1761            405            0.128768         0.147158
     food          S               1691               163            1663            153            0.083900         0.096926
       관리        E|S               1195               301            1110            273            0.084024         0.091385
       소비          S               1024               155             988            146            0.038880         0.042648
       환경          E                896               300             851            281      

## Cell 10. 대학별 · 계열별 · page_type별 요약 및 main/promo 밀도 비교

In [10]:
def group_term_summary(meta_df, count_matrix, group_col, categories_map, tfidf_matrix=None):
    if count_matrix.empty or group_col not in meta_df.columns:
        return pd.DataFrame()

    meta = meta_df[["doc_id", group_col]].copy()
    meta[group_col] = meta[group_col].fillna("missing").astype(str)

    cnt = count_matrix.copy().reset_index()
    merged = meta.merge(cnt, on="doc_id", how="left").fillna(0)
    term_cols = [c for c in merged.columns if c not in {"doc_id", group_col}]
    if not term_cols:
        return pd.DataFrame()

    sum_by_group = merged.groupby(group_col)[term_cols].sum()
    docfreq_by_group = merged.assign(**{t: (merged[t] > 0).astype(int) for t in term_cols}).groupby(group_col)[term_cols].sum()

    tfidf_by_group = None
    if tfidf_matrix is not None and not tfidf_matrix.empty:
        tfidf_df = tfidf_matrix.copy().reset_index()
        tfidf_merged = meta.merge(tfidf_df, on="doc_id", how="inner")
        tfidf_cols = [c for c in tfidf_merged.columns if c not in {"doc_id", group_col}]
        if tfidf_cols:
            tfidf_by_group = tfidf_merged.groupby(group_col)[tfidf_cols].mean()

    records = []
    for grp in sum_by_group.index:
        for term in term_cols:
            rec = {
                group_col: grp,
                "keyword": term,
                "categories": stringify_categories(categories_map.get(term, [])),
                "raw_count": int(sum_by_group.loc[grp, term]),
                "doc_freq": int(docfreq_by_group.loc[grp, term]),
            }
            if tfidf_by_group is not None and grp in tfidf_by_group.index and term in tfidf_by_group.columns:
                rec["mean_tfidf"] = float(tfidf_by_group.loc[grp, term])
            records.append(rec)

    out = pd.DataFrame(records)
    sort_cols = ["raw_count", "doc_freq"] + (["mean_tfidf"] if "mean_tfidf" in out.columns else [])
    return out.sort_values(by=[group_col] + sort_cols, ascending=[True] + [False] * len(sort_cols)).reset_index(drop=True)

def density_summary(df):
    tmp = df.copy()
    tmp["density_per_1k_trackABC"] = np.where(
        tmp["char_len"] > 0,
        tmp["trackABC_hits_total"] / tmp["char_len"] * 1000,
        0.0
    )

    out = tmp.groupby("page_group_resolved", as_index=False).agg(
        n_docs=("doc_id", "nunique"),
        total_chars=("char_len", "sum"),
        trackA_hits_total=("trackA_hits_total", "sum"),
        trackB_exact_hits_total=("trackB_exact_hits_total", "sum"),
        trackB_partial_hits_total=("trackB_partial_hits_total", "sum"),
        trackC_hits_total=("trackC_hits_total", "sum"),
        trackABC_hits_total=("trackABC_hits_total", "sum"),
        mean_doc_density_per_1k=("density_per_1k_trackABC", "mean"),
    )
    out["total_density_per_1k"] = np.where(
        out["total_chars"] > 0,
        out["trackABC_hits_total"] / out["total_chars"] * 1000,
        0.0
    )
    return out.sort_values(by="trackABC_hits_total", ascending=False).reset_index(drop=True)

trackA_by_대학명 = group_term_summary(df, track_a_mat, "대학명", assets["track_a_categories"], tfidf_trackA) if "대학명" in df.columns else pd.DataFrame()
trackA_by_계열 = group_term_summary(df, track_a_mat, "계열", assets["track_a_categories"], tfidf_trackA) if "계열" in df.columns else pd.DataFrame()
trackA_by_page_type = group_term_summary(df, track_a_mat, "page_type", assets["track_a_categories"], tfidf_trackA) if "page_type" in df.columns else pd.DataFrame()

trackB_exact_by_대학명 = group_term_summary(df, track_b_exact_mat, "대학명", assets["track_b_categories"], tfidf_trackB) if "대학명" in df.columns else pd.DataFrame()
trackB_exact_by_계열 = group_term_summary(df, track_b_exact_mat, "계열", assets["track_b_categories"], tfidf_trackB) if "계열" in df.columns else pd.DataFrame()
trackC_by_대학명 = group_term_summary(df, track_c_mat, "대학명", assets["track_c_categories"], tfidf_trackC) if "대학명" in df.columns else pd.DataFrame()

density_main_vs_promo = density_summary(df)

print("[density_main_vs_promo]")
print(density_main_vs_promo.to_string(index=False))

[density_main_vs_promo]
page_group_resolved  n_docs  total_chars  trackA_hits_total  trackB_exact_hits_total  trackB_partial_hits_total  trackC_hits_total  trackABC_hits_total  mean_doc_density_per_1k  total_density_per_1k
               main     866      2883551              21124                      446                      73327                 23                21593                 9.800405              7.488336
              promo     188       250659               1839                       23                       7814                  4                 1866                 8.862090              7.444377


## Cell 11. 저장 — Excel + QA report

In [11]:
keyword_assets_trackA = pd.DataFrame({
    "track_a_vocab_raw": track_a_vocab_raw,
    "categories": [stringify_categories(track_a_categories_raw.get(x, [])) for x in track_a_vocab_raw],
})

keyword_assets_concept = pd.DataFrame({
    "canonical_ko": concept_assets["canonical_vocab"],
    "categories": [stringify_categories(concept_assets["canonical_categories"].get(x, [])) for x in concept_assets["canonical_vocab"]],
})

keyword_assets_trackB = pd.DataFrame({
    "track_b_canonical": assets["track_b_canonical"],
    "compact_key": [normalize_compact(x) for x in assets["track_b_canonical"]],
    "categories": [stringify_categories(assets["track_b_categories"].get(x, [])) for x in assets["track_b_canonical"]],
})

keyword_assets_trackC = pd.DataFrame({
    "track_c_label": list(assets["track_c_patterns"].keys()),
    "pattern": [TRACK_C_REGEX_PATTERNS[k][0] for k in assets["track_c_patterns"].keys()],
    "categories": [stringify_categories(assets["track_c_categories"].get(k, [])) for k in assets["track_c_patterns"].keys()],
})

keyword_dropped_or_excluded = pd.DataFrame({
    "excluded_keywords": pd.Series(assets["excluded_keywords"]),
    "dropped_multis_for_track_a": pd.Series(assets["dropped_multis_for_track_a"]),
    "dropped_english_for_track_a": pd.Series(assets["dropped_english_for_track_a"]),
})

doc_level_debug = df[[
    c for c in [
        "doc_id", "대학명", "단과대학", "계열", "메뉴명", "page_type", "crawl_dual",
        "char_len", "n_nouns_tokens", "n_english_unigram_tokens", "n_trackA_raw_tokens", "n_bigram_tokens", "n_trigram_tokens",
        "trackA_hits_total", "trackA_concept_hits_total", "trackB_exact_hits_total", "trackB_exact_concept_hits_total",
        "trackB_partial_hits_total", "trackC_hits_total", "trackC_concept_hits_total"
    ] if c in df.columns
]].copy()

outputs = {
    "qa_preflight": qa_preflight,
    "trackA_overall_raw": trackA_overall,
    "trackA_overall_concept": trackA_concept_overall,
    "trackA_category_summary_raw": trackA_category_summary,
    "trackA_category_summary_concept": trackA_concept_category_summary,
    "trackB_exact_overall_raw": trackB_exact_overall,
    "trackB_exact_overall_concept": trackB_exact_concept_overall,
    "trackB_partial_overall": trackB_partial_overall,
    "trackB_exact_category_summary_raw": trackB_exact_category_summary,
    "trackB_exact_category_summary_concept": trackB_exact_concept_category_summary,
    "trackB_partial_category_summary": trackB_partial_category_summary,
    "trackC_overall_raw": trackC_overall,
    "trackC_overall_concept": trackC_concept_overall,
    "trackC_category_summary_raw": trackC_category_summary,
    "trackC_category_summary_concept": trackC_concept_category_summary,
    "density_main_vs_promo": density_main_vs_promo,
    "trackA_by_대학명": trackA_by_대학명,
    "trackA_by_계열": trackA_by_계열,
    "trackA_by_page_type": trackA_by_page_type,
    "trackB_exact_by_대학명": trackB_exact_by_대학명,
    "trackB_exact_by_계열": trackB_exact_by_계열,
    "trackC_by_대학명": trackC_by_대학명,
    "keyword_assets_trackA": keyword_assets_trackA,
    "keyword_assets_concept": keyword_assets_concept,
    "keyword_assets_trackB": keyword_assets_trackB,
    "keyword_assets_trackC": keyword_assets_trackC,
    "keyword_dropped_or_excluded": keyword_dropped_or_excluded,
    "doc_level_debug": doc_level_debug,
}

xlsx_path = OUTPUT_DIR / f"빈도분석결과_v14_{INPUT_MODE.lower()}.xlsx"
qa_txt_path = OUTPUT_DIR / f"qa_report_v14_{INPUT_MODE.lower()}.txt"

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    for name, table in outputs.items():
        table.to_excel(writer, sheet_name=safe_sheet_name(name), index=False)

with open(qa_txt_path, "w", encoding="utf-8-sig") as f:
    f.write("[Step II QA Summary]\n")
    for _, row in qa_preflight.iterrows():
        f.write(f"- {row['item']}: {row['value']}\n")

    f.write("\n[Output Sheets]\n")
    for name, table in outputs.items():
        f.write(f"- {name}: rows={len(table)}, cols={len(table.columns)}\n")

print(f"Excel 저장 완료: {xlsx_path}")
print(f"QA report 저장 완료: {qa_txt_path}")


Excel 저장 완료: C:\Users\legen\Desktop\Lab Project\ESG\v2\단계2_빈도분석\빈도분석결과_v14_all.xlsx
QA report 저장 완료: C:\Users\legen\Desktop\Lab Project\ESG\v2\단계2_빈도분석\qa_report_v14_all.txt


## Cell 12. 최종 오류 점검

In [12]:
assert xlsx_path.exists(), "❌ Excel 저장 실패"
assert qa_txt_path.exists(), "❌ QA report 저장 실패"
assert len(trackA_overall) > 0, "❌ Track A 결과가 비어 있음"
assert len(trackB_exact_overall) > 0, "❌ Track B exact 결과가 비어 있음"
assert len(trackC_overall) > 0, "❌ Track C 결과가 비어 있음"
assert len(trackA_concept_overall) > 0, "❌ Track A concept 결과가 비어 있음"
assert len(trackB_exact_concept_overall) > 0, "❌ Track B concept 결과가 비어 있음"
assert len(trackC_concept_overall) > 0, "❌ Track C concept 결과가 비어 있음"
assert "mean_tfidf_overall" in trackA_overall.columns, "❌ TF-IDF 컬럼 누락"

print("[최종 점검 PASS]")
print(f"- 입력 모드: {INPUT_MODE}")
print(f"- 분석 문서 수: {len(df):,}")
print(f"- Track A 키워드 수: {len(assets['track_a_vocab'])}")
print(f"- Track B canonical 수: {len(assets['track_b_canonical'])}")
print(f"- Track C 패턴 수: {len(assets['track_c_patterns'])}")
print(f"- 저장 폴더: {OUTPUT_DIR}")


[최종 점검 PASS]
- 입력 모드: all
- 분석 문서 수: 1,054
- Track A 키워드 수: 110
- Track B canonical 수: 251
- Track C 패턴 수: 6
- 저장 폴더: C:\Users\legen\Desktop\Lab Project\ESG\v2\단계2_빈도분석


## Cell 13. Track A, B, C 맥락 점검

In [13]:
# ============================================================
# [Step II 추가 점검 셀]
# 1) Track A 상위 일반어 문맥 샘플 점검
# 2) Track B exact 저빈도 표현 실제 문장 확인
# 3) Track C 약어 0건 항목 regex 최소 검증
# ============================================================

import re
import ast
import json
import math
import numpy as np
import pandas as pd
from collections import Counter

# ------------------------------------------------------------
# 0. 유틸
# ------------------------------------------------------------
def _find_first_defined(candidates, g=None):
    g = globals() if g is None else g
    for name in candidates:
        if name in g and g[name] is not None:
            return g[name], name
    return None, None

def _safe_parse_list(x):
    if isinstance(x, list):
        return x
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    if isinstance(x, str):
        s = x.strip()
        if s == "":
            return []
        # JSON / Python literal 순으로 시도
        for parser in (json.loads, ast.literal_eval):
            try:
                v = parser(s)
                if isinstance(v, list):
                    return v
            except Exception:
                pass
        # 최후 fallback: 구분자 split
        if "|" in s:
            return [t.strip() for t in s.split("|") if t.strip()]
        if "," in s and s.startswith("[") is False:
            return [t.strip() for t in s.split(",") if t.strip()]
    return []

def _normalize_space(s):
    if s is None:
        return ""
    s = str(s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def _compact_text(s):
    if s is None:
        return ""
    s = str(s)
    s = re.sub(r"[\s_]+", "", s)
    return s.lower()

def _split_sentences(text):
    text = _normalize_space(text)
    if not text:
        return []
    # 문장/줄 단위 분리
    parts = re.split(r'(?<=[\.\?\!。！？])\s+|\n+', text)
    parts = [p.strip() for p in parts if p and p.strip()]
    return parts

def _extract_snippets_by_keyword(text, keyword, max_snippets=3, window=120):
    """
    keyword가 포함된 문장/스니펫 추출
    """
    text = _normalize_space(text)
    if not text:
        return []
    
    sents = _split_sentences(text)
    hits = []
    kw = str(keyword)

    # 1차: 문장 단위
    for sent in sents:
        if kw.lower() in sent.lower():
            hits.append(sent[:300])
        if len(hits) >= max_snippets:
            return hits

    # 2차: 전체 텍스트에서 substring window 추출
    m = re.search(re.escape(kw), text, flags=re.I)
    if m:
        st = max(0, m.start() - window)
        ed = min(len(text), m.end() + window)
        snippet = text[st:ed].strip()
        hits.append(snippet[:300])

    return hits[:max_snippets]

def _extract_snippets_by_phrase_exact(text, phrase, max_snippets=3, window=120):
    """
    Track B exact phrase 점검용
    phrase: '기후_위기' 또는 'supply_chain_management' 등
    compact 기준으로 탐색
    """
    text = _normalize_space(text)
    if not text:
        return []

    compact_full = _compact_text(text)
    compact_phrase = _compact_text(phrase)

    if compact_phrase == "":
        return []

    # 문장 단위 우선
    sents = _split_sentences(text)
    hits = []
    raw_phrase = str(phrase).replace("_", " ")

    for sent in sents:
        if compact_phrase in _compact_text(sent):
            hits.append(sent[:300])
        elif raw_phrase.lower() in sent.lower():
            hits.append(sent[:300])
        if len(hits) >= max_snippets:
            return hits

    # 전체 텍스트 window fallback
    if compact_phrase in compact_full:
        # 원문에서 정확 위치 역추적이 어렵기 때문에
        # phrase의 첫 토큰으로 approximate 검색
        first_tok = str(phrase).split("_")[0]
        m = re.search(re.escape(first_tok), text, flags=re.I)
        if m:
            st = max(0, m.start() - window)
            ed = min(len(text), m.end() + window)
            hits.append(text[st:ed][:300])

    return hits[:max_snippets]

def _infer_text_col(df):
    candidates = [
        'text', 'text_clean', 'clean_text', 'full_text', 'page_text',
        'content', 'body_text', 'raw_text'
    ]
    for c in candidates:
        if c in df.columns:
            return c
    # fallback: object 컬럼 중 가장 긴 텍스트 후보
    obj_cols = [c for c in df.columns if df[c].dtype == 'object']
    if not obj_cols:
        raise ValueError("텍스트 컬럼을 찾지 못했습니다.")
    lens = []
    for c in obj_cols:
        sample = df[c].dropna().astype(str).head(50)
        avg_len = sample.map(len).mean() if len(sample) else 0
        lens.append((c, avg_len))
    lens.sort(key=lambda x: x[1], reverse=True)
    return lens[0][0]

def _infer_doc_id_col(df):
    candidates = ['doc_id', 'document_id', 'id', 'row_id']
    for c in candidates:
        if c in df.columns:
            return c
    return None

def _infer_group_col(df, col_name_candidates):
    for c in col_name_candidates:
        if c in df.columns:
            return c
    return None

def _get_output_df(possible_names):
    """
    globals 직접 탐색 + output dict 탐색
    """
    g = globals()

    # 1) globals에 DataFrame 이름으로 있는 경우
    for name in possible_names:
        if name in g and isinstance(g[name], pd.DataFrame):
            return g[name], name

    # 2) dict 안에 있는 경우
    dict_candidates = ['output_dfs', 'result_dfs', 'sheet_dfs', 'results', 'OUTPUT_DFS']
    for dname in dict_candidates:
        if dname in g and isinstance(g[dname], dict):
            dd = g[dname]
            for name in possible_names:
                if name in dd and isinstance(dd[name], pd.DataFrame):
                    return dd[name], f"{dname}['{name}']"

    return None, None

def _infer_keyword_col(df):
    candidates = ['keyword', 'term', 'token', 'item', 'ngram', 'canonical', 'label']
    for c in candidates:
        if c in df.columns:
            return c
    # 문자열 컬럼 중 첫 번째
    str_cols = [c for c in df.columns if df[c].dtype == 'object']
    if not str_cols:
        raise ValueError("keyword 컬럼을 찾지 못했습니다.")
    return str_cols[0]

def _infer_count_col(df):
    candidates = ['count', 'freq', 'total_count', 'overall_count', 'n']
    for c in candidates:
        if c in df.columns:
            return c
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not num_cols:
        raise ValueError("count 컬럼을 찾지 못했습니다.")
    return num_cols[0]

def _infer_docfreq_col(df):
    candidates = ['doc_freq', 'df', 'doc_n']
    for c in candidates:
        if c in df.columns:
            return c
    return None

def _display_if_exists(df_name, df_obj, n=20):
    print(f"\n[{df_name}]")
    try:
        display(df_obj.head(n))
    except Exception:
        print(df_obj.head(n).to_string(index=False))


In [14]:
# ------------------------------------------------------------
# 1. 메인 데이터프레임 탐색
# ------------------------------------------------------------
df_main, df_main_name = _find_first_defined([
    'df_main', 'df_prepared_main', 'df_prepared', 'df', 'main_df'
])

if df_main is None or not isinstance(df_main, pd.DataFrame):
    raise ValueError("메인 DataFrame(df_main 등)을 찾지 못했습니다. 단계2 전처리/prepare_dataframe 결과를 먼저 확인하세요.")

text_col = _infer_text_col(df_main)
doc_id_col = _infer_doc_id_col(df_main)

univ_col = _infer_group_col(df_main, ['대학명', 'university', 'school_name'])
track_col = _infer_group_col(df_main, ['계열', 'track', 'series'])
page_type_col = _infer_group_col(df_main, ['page_type', 'pageType'])
page_category_col = _infer_group_col(df_main, ['page_category', 'category'])

# list 컬럼 준비
list_cols_needed = [
    'trackA_raw_total_list',
    'english_unigrams_filtered',
    'nouns_filtered',
    'sent_bigrams_list',
    'sent_trigrams_list'
]
for c in list_cols_needed:
    if c in df_main.columns:
        df_main[c] = df_main[c].apply(_safe_parse_list)

# Track A raw 토큰 보정
if 'trackA_raw_total_list' not in df_main.columns:
    if 'nouns_filtered' in df_main.columns and 'english_unigrams_filtered' in df_main.columns:
        df_main['trackA_raw_total_list'] = df_main['nouns_filtered'].apply(_safe_parse_list) + df_main['english_unigrams_filtered'].apply(_safe_parse_list)
    elif 'nouns_filtered' in df_main.columns:
        df_main['trackA_raw_total_list'] = df_main['nouns_filtered'].apply(_safe_parse_list)
    else:
        raise ValueError("trackA_raw_total_list 또는 nouns_filtered 컬럼이 없습니다.")

In [15]:
# ------------------------------------------------------------
# 2. 결과표 DataFrame 탐색
# ------------------------------------------------------------
trackA_raw_df, trackA_raw_df_name = _get_output_df([
    'trackA_overall_raw'
])

trackB_exact_raw_df, trackB_exact_raw_df_name = _get_output_df([
    'trackB_exact_overall_raw'
])

trackC_raw_df, trackC_raw_df_name = _get_output_df([
    'trackC_overall_raw'
])

# 결과표가 없으면 최소 계산
if trackA_raw_df is None:
    tmp_counter = Counter()
    doc_counter = Counter()
    for toks in df_main['trackA_raw_total_list']:
        toks = _safe_parse_list(toks)
        tmp_counter.update(toks)
        doc_counter.update(set(toks))
    trackA_raw_df = pd.DataFrame({
        'keyword': list(tmp_counter.keys()),
        'count': list(tmp_counter.values()),
        'doc_freq': [doc_counter[k] for k in tmp_counter.keys()]
    }).sort_values(['count', 'doc_freq', 'keyword'], ascending=[False, False, True]).reset_index(drop=True)
    trackA_raw_df_name = 'computed_trackA_overall_raw'

if trackB_exact_raw_df is None:
    bg_counter = Counter()
    tg_counter = Counter()
    bg_doc = Counter()
    tg_doc = Counter()

    if 'sent_bigrams_list' in df_main.columns:
        for toks in df_main['sent_bigrams_list']:
            toks = _safe_parse_list(toks)
            bg_counter.update(toks)
            bg_doc.update(set(toks))

    if 'sent_trigrams_list' in df_main.columns:
        for toks in df_main['sent_trigrams_list']:
            toks = _safe_parse_list(toks)
            tg_counter.update(toks)
            tg_doc.update(set(toks))

    merged_keys = set(bg_counter.keys()) | set(tg_counter.keys())
    trackB_exact_raw_df = pd.DataFrame({
        'keyword': list(merged_keys),
        'count': [bg_counter[k] + tg_counter[k] for k in merged_keys],
        'doc_freq': [bg_doc[k] + tg_doc[k] for k in merged_keys]
    }).sort_values(['count', 'doc_freq', 'keyword'], ascending=[False, False, True]).reset_index(drop=True)
    trackB_exact_raw_df_name = 'computed_trackB_exact_overall_raw'

if trackC_raw_df is None:
    # fallback pattern set
    trackC_raw_df = pd.DataFrame({
        'keyword': ['ESG', 'GHG', 'GRI', 'ISSB', 'SASB', 'TCFD'],
        'count': [np.nan]*6,
        'doc_freq': [np.nan]*6
    })
    trackC_raw_df_name = 'fallback_trackC_overall_raw'

# 컬럼명 추론
a_kw_col = _infer_keyword_col(trackA_raw_df)
a_cnt_col = _infer_count_col(trackA_raw_df)
a_df_col = _infer_docfreq_col(trackA_raw_df)

b_kw_col = _infer_keyword_col(trackB_exact_raw_df)
b_cnt_col = _infer_count_col(trackB_exact_raw_df)
b_df_col = _infer_docfreq_col(trackB_exact_raw_df)

c_kw_col = _infer_keyword_col(trackC_raw_df)
c_cnt_col = _infer_count_col(trackC_raw_df)
c_df_col = _infer_docfreq_col(trackC_raw_df)

print(f"[INFO] df_main: {df_main_name}, text_col: {text_col}")
print(f"[INFO] TrackA raw df: {trackA_raw_df_name}")
print(f"[INFO] TrackB exact raw df: {trackB_exact_raw_df_name}")
print(f"[INFO] TrackC raw df: {trackC_raw_df_name}")


[INFO] df_main: df, text_col: text
[INFO] TrackA raw df: computed_trackA_overall_raw
[INFO] TrackB exact raw df: computed_trackB_exact_overall_raw
[INFO] TrackC raw df: fallback_trackC_overall_raw


In [16]:
# ------------------------------------------------------------
# 3. Track A 상위 일반어 문맥 샘플 점검
# ------------------------------------------------------------
TOP_N_TRACKA = 15
SNIPPETS_PER_KEYWORD = 3

trackA_top = trackA_raw_df.copy()
trackA_top = trackA_top.sort_values(a_cnt_col, ascending=False).head(TOP_N_TRACKA).reset_index(drop=True)

trackA_context_rows = []

for _, row in trackA_top.iterrows():
    kw = str(row[a_kw_col])
    cnt = row[a_cnt_col]
    dfreq = row[a_df_col] if a_df_col is not None and a_df_col in row else np.nan

    hit_df = df_main[df_main['trackA_raw_total_list'].apply(lambda xs: kw in _safe_parse_list(xs))].copy()
    if hit_df.empty:
        trackA_context_rows.append({
            'keyword': kw,
            'count': cnt,
            'doc_freq': dfreq,
            'sample_doc_n': 0,
            'sample_1': None,
            'sample_2': None,
            'sample_3': None
        })
        continue

    # 텍스트 길이 긴 것/앞쪽 문서 우선이 아니라, 단순 head로 샘플
    samples = []
    for _, r in hit_df.head(20).iterrows():
        snippets = _extract_snippets_by_keyword(r[text_col], kw, max_snippets=1)
        if snippets:
            samples.append(snippets[0])
        if len(samples) >= SNIPPETS_PER_KEYWORD:
            break

    trackA_context_rows.append({
        'keyword': kw,
        'count': cnt,
        'doc_freq': dfreq,
        'sample_doc_n': len(hit_df),
        'sample_1': samples[0] if len(samples) > 0 else None,
        'sample_2': samples[1] if len(samples) > 1 else None,
        'sample_3': samples[2] if len(samples) > 2 else None
    })

trackA_context_check = pd.DataFrame(trackA_context_rows)

In [17]:
# ------------------------------------------------------------
# 4. Track B exact 저빈도 표현 실제 문장 확인
# ------------------------------------------------------------
LOW_FREQ_MAX = 2
LOW_FREQ_TOP_N = 30
SNIPPETS_PER_PHRASE = 2

trackB_low = trackB_exact_raw_df.copy()
trackB_low = trackB_low[trackB_low[b_cnt_col].fillna(0) > 0].copy()
trackB_low = trackB_low[trackB_low[b_cnt_col] <= LOW_FREQ_MAX].copy()
trackB_low = trackB_low.sort_values([b_cnt_col, b_kw_col], ascending=[True, True]).head(LOW_FREQ_TOP_N).reset_index(drop=True)

trackB_lowfreq_rows = []

# Track B source list 결합
def _trackB_all_ngrams(row):
    out = []
    if 'sent_bigrams_list' in row.index:
        out.extend(_safe_parse_list(row['sent_bigrams_list']))
    if 'sent_trigrams_list' in row.index:
        out.extend(_safe_parse_list(row['sent_trigrams_list']))
    return out

for _, row in trackB_low.iterrows():
    phrase = str(row[b_kw_col])
    cnt = row[b_cnt_col]
    dfreq = row[b_df_col] if b_df_col is not None and b_df_col in row else np.nan
    compact_phrase = _compact_text(phrase)

    hit_idx = []
    for i, r in df_main.iterrows():
        ngrams = _trackB_all_ngrams(r)
        found = any(_compact_text(x) == compact_phrase for x in ngrams)
        if not found:
            # 텍스트 compact fallback
            found = compact_phrase in _compact_text(r[text_col])
        if found:
            hit_idx.append(i)
        if len(hit_idx) >= 10:
            break

    samples = []
    for idx in hit_idx:
        txt = df_main.loc[idx, text_col]
        snippets = _extract_snippets_by_phrase_exact(txt, phrase, max_snippets=1)
        if snippets:
            samples.append(snippets[0])
        if len(samples) >= SNIPPETS_PER_PHRASE:
            break

    trackB_lowfreq_check_rows = {
        'phrase': phrase,
        'count': cnt,
        'doc_freq': dfreq,
        'sample_doc_n_found': len(hit_idx),
        'sample_1': samples[0] if len(samples) > 0 else None,
        'sample_2': samples[1] if len(samples) > 1 else None
    }
    trackB_lowfreq_rows.append(trackB_lowfreq_check_rows)

trackB_lowfreq_check = pd.DataFrame(trackB_lowfreq_rows)

In [18]:
# ------------------------------------------------------------
# 5. Track C 약어 0건 항목 regex 최소 검증
# ------------------------------------------------------------
# notebook 내 정의된 pattern 자산이 있으면 사용, 없으면 fallback
fallback_trackc_patterns = {
    'ESG':  r'(?<![A-Za-z])ESG(?![A-Za-z])',
    'GHG':  r'(?<![A-Za-z])GHG(?![A-Za-z])',
    'GRI':  r'(?<![A-Za-z])GRI(?![A-Za-z])',
    'ISSB': r'(?<![A-Za-z])ISSB(?![A-Za-z])',
    'SASB': r'(?<![A-Za-z])SASB(?![A-Za-z])',
    'TCFD': r'(?<![A-Za-z])TCFD(?![A-Za-z])',
}

pattern_asset_candidates = [
    'TRACK_C_PATTERNS', 'track_c_patterns',
    'TRACKC_PATTERNS', 'trackC_patterns',
    'ABBR_PATTERNS', 'abbr_patterns'
]

trackc_patterns = None
for nm in pattern_asset_candidates:
    if nm in globals() and isinstance(globals()[nm], dict):
        trackc_patterns = globals()[nm]
        break

if trackc_patterns is None:
    trackc_patterns = fallback_trackc_patterns

trackC_zero = trackC_raw_df.copy()
trackC_zero = trackC_zero[trackC_zero[c_cnt_col].fillna(0) == 0].copy()

trackC_regex_check_rows = []

for _, row in trackC_zero.iterrows():
    kw = str(row[c_kw_col])

    # pattern 찾기
    pattern = None
    if kw in trackc_patterns:
        pattern = trackc_patterns[kw]
    else:
        # key normalize fallback
        for k, v in trackc_patterns.items():
            if str(k).strip().upper() == kw.strip().upper():
                pattern = v
                break
    if pattern is None:
        pattern = fallback_trackc_patterns.get(kw.upper(), rf'(?<![A-Za-z]){re.escape(kw)}(?![A-Za-z])')

    doc_hits = 0
    snippets = []

    try:
        rgx = re.compile(pattern, flags=re.I)
    except re.error:
        # regex가 잘못되어 있으면 literal fallback
        rgx = re.compile(re.escape(kw), flags=re.I)

    for _, r in df_main.iterrows():
        txt = _normalize_space(r[text_col])
        if not txt:
            continue
        if rgx.search(txt):
            doc_hits += 1
            snips = _extract_snippets_by_keyword(txt, kw, max_snippets=1)
            if snips:
                snippets.append(snips[0])
            if len(snippets) >= 3:
                break

    trackC_regex_check_rows.append({
        'abbr': kw,
        'reported_count_in_trackC_raw': row[c_cnt_col],
        'independent_regex_doc_hits': doc_hits,
        'regex_pattern_used': pattern,
        'sample_1': snippets[0] if len(snippets) > 0 else None,
        'sample_2': snippets[1] if len(snippets) > 1 else None,
        'sample_3': snippets[2] if len(snippets) > 2 else None,
        'qa_flag': 'POTENTIAL_REGEX_REVIEW' if doc_hits > 0 else 'LIKELY_TRUE_ZERO'
    })

trackC_regex_zero_check = pd.DataFrame(trackC_regex_check_rows)

In [19]:
# ------------------------------------------------------------
# 6. 요약표
# ------------------------------------------------------------
summary_rows = []

summary_rows.append({
    'check_item': 'TrackA_top_context_check',
    'n_rows': len(trackA_context_check),
    'note': f'Top {TOP_N_TRACKA} keywords, up to {SNIPPETS_PER_KEYWORD} snippets each'
})

summary_rows.append({
    'check_item': 'TrackB_lowfreq_exact_check',
    'n_rows': len(trackB_lowfreq_check),
    'note': f'count <= {LOW_FREQ_MAX}, up to {SNIPPETS_PER_PHRASE} snippets each'
})

summary_rows.append({
    'check_item': 'TrackC_zero_regex_check',
    'n_rows': len(trackC_regex_zero_check),
    'note': 'independent regex verification for zero-count abbreviations'
})

qa_step3_precheck_summary = pd.DataFrame(summary_rows)

# ------------------------------------------------------------
# 7. 출력
# ------------------------------------------------------------
print("\n" + "="*70)
print("[Step II 추가 점검 완료]")
print("="*70)

_display_if_exists("qa_step3_precheck_summary", qa_step3_precheck_summary, n=20)
_display_if_exists("trackA_context_check", trackA_context_check, n=20)
_display_if_exists("trackB_lowfreq_check", trackB_lowfreq_check, n=30)
_display_if_exists("trackC_regex_zero_check", trackC_regex_zero_check, n=20)

print("\n[참고]")
print("- trackA_context_check: 상위 unigram의 실제 문맥 샘플")
print("- trackB_lowfreq_check: 저빈도 exact phrase의 실제 문장 샘플")
print("- trackC_regex_zero_check: 0건 약어의 독립 regex 검증 결과")
print("- 필요하면 이 3개 DataFrame을 그대로 복사해 보고서/메모에 사용하면 됩니다.")


[Step II 추가 점검 완료]

[qa_step3_precheck_summary]


,check_item,n_rows,note
0,TrackA_top_context_check,15,"Top 15 keywords, up to 3 snippets each"
1,TrackB_lowfreq_exact_check,30,"count <= 2, up to 2 snippets each"
2,TrackC_zero_regex_check,6,independent regex verification for zero-count ...



[trackA_context_check]


,keyword,count,doc_freq,sample_doc_n,sample_1,sample_2,sample_3
0,식품,4477,359,359,"222동 427호 분야 (저축 및 , , , , 등) 금융소비자문제와 대응방안 재무...",[석사졸업]신정아 이메일: 직장/직위: LG전자/ 원 졸업논문: 석) 전문홈쇼핑채널...,소비자업무전문가 등록민간자격 2013-1513 구 분 교 과 목 대체인정교과목 이수...
1,한국,3588,381,381,"젝트 폴리탄 공간· 장소 트렌드 조사, 한국토지주택공사 원, 2023 2024.01...",빅데이터 분석기사는 한국데이터산업진흥원에서 시행하는 정형/비정형 대용량 데이터의 구...,"222동 427호 분야 (저축 및 , , , , 등) 금융소비자문제와 대응방안 재무..."
2,교육,3539,557,557,"젝트 폴리탄 공간· 장소 트렌드 조사, 한국토지주택공사 원, 2023 2024.01...",학과소개 [소비자학의 교육과정] 소비자학은 소비자의 복지 향상과 건전한 소비문화 형...,소비자업무전문가 소비자업무전문가는 소비자와 관련된 제반 업무를 기업과 소비자보호관련...
3,소비자,3167,223,223,학과소개 [소비자학의 교육과정] 소비자학은 소비자의 복지 향상과 건전한 소비문화 형...,소비자학과에서는 학부생이 소비자학 필요한 역량을 개발할 수 록 다양한 을 진행하고 ...,"소비자전문상담사 소비자전문상담사는 소비자들의 불만 해결 및 제, 소비자 의사결정을 ..."
4,디자인,2822,294,294,"아울러 소비자경제학과의 교수진은 소비자경제학뿐만 아니라 마케팅, 금융, 사용자경험디...",빅데이터와소비자트렌드(캡스톤디자인) 본 과목은 장에서의 소비문화를 살피는데 있어 빅...,전공 내 특화된 트랙을 교수의 관리 하에 이수하는 맞춤형 전공교육 각 트랙별 설정교...
5,패션,2797,174,174,중국 Z세대 소비자의 감정적 소비 제품에 대한 지불의사 : 2026 학위 : 석사 ...,HEE 527 패션미학특론(Special Topics in Aesthetics of...,"대학원 생활과학과에서는 학, 패션디자인 및 머천다이징, 식품영양학, 소비자학, 가정..."
6,대학,2211,406,406,"젝트 폴리탄 공간· 장소 트렌드 조사, 한국토지주택공사 원, 2023 2024.01...",arrow-left 소비자학과 대학원 실 소비자정보⋅ 실 담당교수 나종연 교수 소비...,arrow-left 소비자학과 대학원 실 실 담당교수 여정성 교수 실에서는 의 분석...
7,분석,2146,395,395,학부 3학년 학생들이 주축이 되는 학부 심포지엄팀은 최근의 소비자 관련 이슈에 대하...,소비자업무전문가 소비자업무전문가는 소비자와 관련된 제반 업무를 기업과 소비자보호관련...,arrow-left 소비자학과 대학원 실 소비자정보⋅ 실 담당교수 나종연 교수 소비...
8,사회,1894,430,430,대량생산과 대량소비가 이루어지는 현대 산업사회에서 소비자문제는 중요한 사회적 대두하...,"사회조사분석사 사회조사분석사란 국가 및 지방자치단체, 기업, 정당, 사회단체 등에서...","222동 427호 분야 (저축 및 , , , , 등) 금융소비자문제와 대응방안 재무..."
9,개발,1808,435,435,"젝트 폴리탄 공간· 장소 트렌드 조사, 한국토지주택공사 원, 2023 2024.01...",소비자학과에서는 학부생이 소비자학 필요한 역량을 개발할 수 록 다양한 을 진행하고 ...,"222동 427호 분야 (저축 및 , , , , 등) 금융소비자문제와 대응방안 재무..."



[trackB_lowfreq_check]


,phrase,count,doc_freq,sample_doc_n_found,sample_1,sample_2
0,AAIDD_설립,1,1,1,None,None
1,AAIDD_설립_역사,1,1,1,None,None
2,AAIDD_지난,1,1,1,None,None
3,AAIDD_지난_공식,1,1,1,None,None
4,AAIDD_초청,1,1,1,None,None
5,AAIDD_초청_증서,1,1,1,None,None
6,AATCC_인증_업계,1,1,1,None,None
7,ABA_발달,1,1,1,• 가족센터 • (주) 비교육 • (주) 레몬트리 • 청소년상담센터 • 가족센터 •...,None
8,ABA_발달_교실,1,1,1,• 가족센터 • (주) 비교육 • (주) 레몬트리 • 청소년상담센터 • 가족센터 •...,None
9,ABA_이해,1,1,1,ABA이해와 적용.,None



[trackC_regex_zero_check]


,abbr,reported_count_in_trackC_raw,independent_regex_doc_hits,regex_pattern_used,sample_1,sample_2,sample_3,qa_flag
0,ESG,NaN,3,(?<![A-Za-z])ESG(?![A-Za-z]),“공정성 가치와 소비자 시민성.” [석사과정] 현 이메일: 관심분야: 소비자데이터분...,민성이 금융기관의 ESG 평가에 미치는 영향.,중국 Z세대 소비자의 감정적 소비 제품에 대한 지불의사 : 2026 학위 : 석사 ...,POTENTIAL_REGEX_REVIEW
1,GHG,NaN,0,(?<![A-Za-z])GHG(?![A-Za-z]),None,None,None,LIKELY_TRUE_ZERO
2,GRI,NaN,0,(?<![A-Za-z])GRI(?![A-Za-z]),None,None,None,LIKELY_TRUE_ZERO
3,ISSB,NaN,0,(?<![A-Za-z])ISSB(?![A-Za-z]),None,None,None,LIKELY_TRUE_ZERO
4,SASB,NaN,0,(?<![A-Za-z])SASB(?![A-Za-z]),None,None,None,LIKELY_TRUE_ZERO
5,TCFD,NaN,0,(?<![A-Za-z])TCFD(?![A-Za-z]),None,None,None,LIKELY_TRUE_ZERO



[참고]
- trackA_context_check: 상위 unigram의 실제 문맥 샘플
- trackB_lowfreq_check: 저빈도 exact phrase의 실제 문장 샘플
- trackC_regex_zero_check: 0건 약어의 독립 regex 검증 결과
- 필요하면 이 3개 DataFrame을 그대로 복사해 보고서/메모에 사용하면 됩니다.


In [20]:
qa_step3_precheck_summary.to_csv(OUTPUT_DIR / f"qa_step3_precheck_summary_{INPUT_MODE.lower()}.csv", index=False, encoding='utf-8-sig')
trackA_context_check.to_csv(OUTPUT_DIR / f"trackA_context_check_{INPUT_MODE.lower()}.csv", index=False, encoding='utf-8-sig')
trackB_lowfreq_check.to_csv(OUTPUT_DIR / f"trackB_lowfreq_check_{INPUT_MODE.lower()}.csv", index=False, encoding='utf-8-sig')
trackC_regex_zero_check.to_csv(OUTPUT_DIR / f"trackC_regex_zero_check_{INPUT_MODE.lower()}.csv", index=False, encoding='utf-8-sig')